# 02 · Preparación para el modelo

Separar train y test, seleccionar variables y transformarlas a números. Todo lo que depende del target se decide solo con train.

In [1]:
import sys
sys.path.append('..')

import pandas as pd
from scipy.stats import chi2_contingency, mannwhitneyu
from sklearn.model_selection import train_test_split

from src.preprocesamiento import COLUMNAS, crear_preprocesador

df = pd.read_csv('../data/processed/reclamaciones_limpio.csv')
df.shape

(15419, 29)

## 1. Separación train / test

80 % train y 20 % test, estratificado para mantener el 6 % de fraude.

In [2]:
train, test = train_test_split(df, test_size=0.2, stratify=df['FraudFound_P'], random_state=42)

print(len(train), len(test))
print('Fraude en train:', train['FraudFound_P'].mean().round(4), '| en test:', test['FraudFound_P'].mean().round(4))

train.to_csv('../data/processed/train.csv', index=False)
test.to_csv('../data/processed/test.csv', index=False)

12335 3084
Fraude en train: 0.0598 | en test: 0.06


Train y test mantienen la misma proporción de fraude (6 %).

## 2. Selección de variables

Test chi² contra el fraude (Mann-Whitney para Age), solo con train. Un p-valor mayor que 0,05 indica que no hay evidencia de relación.

In [3]:
categoricas = train.columns.drop(['FraudFound_P', 'Age'])
p_valores = pd.Series({c: chi2_contingency(pd.crosstab(train[c], train['FraudFound_P']))[1] for c in categoricas})

edad = train.dropna(subset=['Age'])
p_valores['Age'] = mannwhitneyu(edad.loc[edad['FraudFound_P'] == 1, 'Age'], edad.loc[edad['FraudFound_P'] == 0, 'Age']).pvalue

p_valores.sort_values().round(4)

BasePolicy              0.0000
VehicleCategory         0.0000
Fault                   0.0000
AddressChange_Claim     0.0000
Deductible              0.0000
PastNumberOfClaims      0.0000
VehiclePrice            0.0000
Make                    0.0000
Month                   0.0005
retraso_meses           0.0006
MonthClaimed            0.0009
Sex                     0.0011
AccidentArea            0.0027
AgentType               0.0051
AgeOfPolicyHolder       0.0066
Days_Policy_Accident    0.0160
AgeOfVehicle            0.0246
Days_Policy_Claim       0.0409
Age                     0.0484
DayOfWeek               0.3852
WitnessPresent          0.4213
RepNumber               0.4548
WeekOfMonth             0.5541
WeekOfMonthClaimed      0.6814
DayOfWeekClaimed        0.7703
NumberOfCars            0.8262
DriverRating            0.8334
MaritalStatus           0.8856
dtype: float64

Decisión: conservar las variables con p < 0,05 y eliminar las otras 9: DayOfWeek, WitnessPresent, RepNumber, WeekOfMonth, WeekOfMonthClaimed, DayOfWeekClaimed, NumberOfCars, DriverRating y MaritalStatus.

Con 28 tests, 1 o 2 pueden salir significativas por azar; las menos seguras son las de p entre 0,01 y 0,05 (Days_Policy_Accident, AgeOfVehicle, Days_Policy_Claim y Age).

Comprobación contra src/preprocesamiento.py:

In [4]:
conservadas = set(p_valores[p_valores < 0.05].index)
print(conservadas == set(COLUMNAS))
print('Eliminadas:', sorted(set(p_valores.index) - conservadas))

True
Eliminadas: ['DayOfWeek', 'DayOfWeekClaimed', 'DriverRating', 'MaritalStatus', 'NumberOfCars', 'RepNumber', 'WeekOfMonth', 'WeekOfMonthClaimed', 'WitnessPresent']


## 3. Transformación a números

Ordinales con orden natural, one-hot para las demás (categorías con menos de 100 casos en train se agrupan en Otras) y numéricas sin tocar. El preprocesador se ajusta solo con train.

In [5]:
X_train = train[COLUMNAS]
X_test = test[COLUMNAS]

preprocesador = crear_preprocesador()
X_train_num = preprocesador.fit_transform(X_train)
X_test_num = preprocesador.transform(X_test)

print(X_train_num.shape, X_test_num.shape)

(12335, 64) (3084, 64)


Categorías agrupadas en Otras:

In [6]:
onehot = preprocesador.named_transformers_['nominales']
for columna, raras in zip(onehot.feature_names_in_, onehot.infrequent_categories_):
    if raras is not None:
        print(columna, '->', list(raras))

Make -> ['BMW', 'Dodge', 'Ferrari', 'Jaguar', 'Mercedes', 'Mercury', 'Nissan', 'Porsche', 'Saab', 'Saturn']
Deductible -> [np.int64(300)]
AddressChange_Claim -> ['under 6 months']


Datos listos: matrices numéricas sin fugas. El preprocesador se incluirá dentro del Pipeline de cada modelo en el notebook 03.